# 18.6 高斯过程 / Gaussian Processes

**中文**：18.1 的贝叶斯线性回归学的是**权重的分布**——但它只能拟合直线(或你手工指定的基函数)。**高斯过程(Gaussian Process, GP)** 是一个惊人优雅的飞跃:它不学参数,而是**直接在"函数"上定义一个概率分布**。你不用再猜"该用几次多项式、什么基函数",GP 让**数据自己决定函数长什么样**,还自带**精准的不确定性**。一句话:*"GP = 无穷多基函数的贝叶斯回归"*,是**非参数**贝叶斯的代表,也是贝叶斯优化(下节)的引擎。
**English**: Bayesian linear regression (18.1) learned a **distribution over weights** — but it only fits lines (or hand-picked basis functions). A **Gaussian Process (GP)** is a strikingly elegant leap: instead of parameters, it **defines a probability distribution directly over "functions."** No more guessing "which polynomial degree, which basis functions" — a GP lets **the data decide what the function looks like**, with **precise built-in uncertainty**. In one line: *"GP = Bayesian regression with infinitely many basis functions,"* the flagship of **non-parametric** Bayes and the engine of Bayesian optimization (next).

---

**中文**：怎么给"函数"定义分布？关键洞察:一个函数就是"每个输入 $x$ 对应一个输出 $f(x)$"。GP 说:**任意有限个点上的函数值,联合起来服从一个多元高斯分布**。这个高斯由两样东西完全决定:
**English**: How to define a distribution over "functions"? Key insight: a function is "each input $x$ maps to an output $f(x)$." A GP says: **the function values at any finite set of points are jointly a multivariate Gaussian**. This Gaussian is fully determined by two things:
- **均值函数 $m(x)$**(常设为 0):函数的先验平均水平。
  **Mean function $m(x)$** (often set to 0): the function's prior average level.
- **核函数(协方差函数)$k(x,x')$**:**两点的输出有多相关**——这是 GP 的灵魂。最常用的 **RBF 核**:$k(x,x')=\sigma_f^2\exp(-\frac{\|x-x'\|^2}{2\ell^2})$,意思是"**两点越近,函数值越相关**"。**长度尺度 $\ell$** 控制函数变化的快慢(平滑度)。

  **Kernel (covariance function) $k(x,x')$**: **how correlated two points' outputs are** — the soul of a GP. The common **RBF kernel**: $k(x,x')=\sigma_f^2\exp(-\frac{\|x-x'\|^2}{2\ell^2})$, meaning "**the closer two points, the more correlated their values**." The **lengthscale $\ell$** controls how fast the function varies (smoothness).

**中文**：有了先验(核),**预测就是高斯条件分布的闭式公式**。给定观测 $(X,\mathbf y)$,新点 $X_*$ 的预测:
**English**: Given the prior (kernel), **prediction is the closed-form Gaussian conditional**. Given observations $(X,\mathbf y)$, for new points $X_*$:

$$\boldsymbol\mu_* = K_{*}\,[K+\sigma_n^2 I]^{-1}\mathbf y,\qquad \boldsymbol\Sigma_* = K_{**}-K_{*}\,[K+\sigma_n^2 I]^{-1}K_{*}^\top$$

**中文**：$K$ 是训练点两两核值矩阵,$K_*$ 是新点与训练点的核值,$\sigma_n^2$ 是观测噪声。预测**均值**是训练标签的加权平均(权重由核给出="离得近的点说话更响"),预测**协方差**给出每个点的不确定性——**离数据越远,第二项越小,方差越接近先验(变大)**。这就是 GP 最迷人的特性:**在没有数据的地方,它诚实地说"我不知道"**。
**English**: $K$ is the train-train kernel matrix, $K_*$ the train-new kernel values, $\sigma_n^2$ observation noise. The predictive **mean** is a weighted average of training labels (weights from the kernel = "nearby points speak louder"), and the predictive **covariance** gives per-point uncertainty — **the farther from data, the smaller the second term, so variance approaches the prior (grows)**. This is a GP's most beautiful trait: **where there is no data, it honestly says "I don't know."**

> 💡 **面试速查 / Interview cheat-sheet（★★★ 非参数贝叶斯必考）**
> **中文**：**GP=函数上的分布**:任意有限点的函数值服从多元高斯, 由均值(常0)+**核**决定。核=两点相关性→控制平滑度(**长度尺度 ℓ**:小→摆动快, 大→平滑)。**预测=高斯条件公式(闭式)**, 给均值 + **随距离增长的方差**(数据外自动变不确定)。**非参数**(复杂度随数据长)。**超参(ℓ, σ_f, σ_n)** 用**边际似然(marginal likelihood)** 最大化自动选(内置奥卡姆剃刀)。**核心痛点**:$[K+\sigma_n^2 I]^{-1}$ 是 $O(n^3)$、存 $O(n^2)$——大数据要**稀疏 GP/诱导点**。核选择编码先验(RBF 光滑、Matérn 粗糙、周期核…)。应用:小数据回归+不确定性、**贝叶斯优化**、地统计(kriging)、时序。
> **English**: **GP = a distribution over functions**: function values at any finite points are jointly Gaussian, set by a mean (usually 0) + a **kernel**. The kernel = correlation between points → controls smoothness (**lengthscale ℓ**: small → wiggly, large → smooth). **Prediction = the closed-form Gaussian conditional**, giving a mean + **variance that grows with distance** (auto-uncertain outside data). **Non-parametric** (complexity grows with data). **Hyperparameters (ℓ, σ_f, σ_n)** are chosen automatically by maximizing the **marginal likelihood** (a built-in Occam's razor). **Key pain**: $[K+\sigma_n^2 I]^{-1}$ is $O(n^3)$ time, $O(n^2)$ memory — big data needs **sparse GPs / inducing points**. The kernel encodes the prior (RBF smooth, Matérn rougher, periodic…). Uses: small-data regression + uncertainty, **Bayesian optimization**, geostatistics (kriging), time series.


In [ ]:

# ============================================================
# 从零实现 RBF 核 + 从 GP 先验采样函数 / RBF kernel + sample functions from the GP prior
# 中文:先验(还没看数据)就是"所有平滑函数上的分布"。我们从先验里采样几条函数——它们都平滑, 但形状各异。
# English: the prior (before data) is "a distribution over all smooth functions." Sample a few — all smooth, varied.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
np.random.seed(1)
def rbf_kernel(A, B, l=1.0, sf=1.0):                          # RBF/高斯核 / squared-exponential kernel
    d = np.sum(A**2,1)[:,None] + np.sum(B**2,1)[None,:] - 2*A@B.T   # 两两平方距离 / pairwise sq-dist
    return sf**2 * np.exp(-0.5*d/l**2)

Xs = np.linspace(-6, 6, 200).reshape(-1,1)                   # 测试网格 / test grid
K_prior = rbf_kernel(Xs, Xs, l=1.0, sf=1.0) + 1e-9*np.eye(len(Xs))
prior_samples = np.random.multivariate_normal(np.zeros(len(Xs)), K_prior, size=5)   # 采样5条函数 / sample
print("从 GP 先验采样了 5 条函数(都平滑, 均值0, 幅度~1)/ 5 sampled prior functions")
print("核的含义:近的点强相关(k≈1), 远的点几乎独立(k≈0) / kernel: near points correlated, far ~independent")


**中文**：现在给一些**观测数据**,GP 会把先验"收窄"成**后验**——只保留那些**穿过数据点**的函数。我们故意在中间留一个**数据空缺**,并让预测区间超出数据两端,好观察 GP 如何在"没数据的地方"张开不确定性。用**边际似然**自动选长度尺度等超参。
**English**: Now with some **observed data**, the GP narrows the prior into a **posterior** — keeping only functions that **pass through the data**. We deliberately leave a **gap** in the middle and predict beyond both ends, to watch the GP widen uncertainty "where there is no data." Hyperparameters (lengthscale etc.) are chosen automatically by the **marginal likelihood**.


In [ ]:

# ============================================================
# GP 后验:条件于数据的闭式公式 + 边际似然选超参 / GP posterior + marginal-likelihood hyperparams
# ============================================================
def true_f(x): return np.sin(1.5*x)                          # 真实函数(未知)/ true function
X = np.array([-4,-3.2,-2.5,-1.8,-1.0, 1.7,2.5,3.3,3.9]).reshape(-1,1)   # 注意中间[-1,1.7]有空缺 / gap
y = true_f(X).ravel() + np.random.normal(0, 0.08, len(X))    # 带噪观测 / noisy observations

def log_marginal_likelihood(l, sf, sn):                      # 边际似然(自动奥卡姆剃刀)/ marginal likelihood
    K = rbf_kernel(X,X,l,sf) + sn**2*np.eye(len(X))
    try: L=np.linalg.cholesky(K)
    except: return -1e9
    a=np.linalg.solve(L.T, np.linalg.solve(L,y))
    return -0.5*y@a - np.sum(np.log(np.diag(L))) - 0.5*len(X)*np.log(2*np.pi)

# 网格搜最大边际似然的超参 / grid-search hyperparameters maximizing marginal likelihood
best=(-1e9,None)
for l in np.linspace(0.3,3,28):
    for sf in np.linspace(0.5,2,16):
        for sn in [0.05,0.08,0.15]:
            v=log_marginal_likelihood(l,sf,sn)
            if v>best[0]: best=(v,(l,sf,sn))
l_opt,sf_opt,sn_opt = best[1]
print(f"边际似然选出的超参 / MLL-selected: lengthscale={l_opt:.2f}, sf={sf_opt:.2f}, noise={sn_opt}")

def gp_posterior(l, sf, sn):
    K  = rbf_kernel(X,X,l,sf) + sn**2*np.eye(len(X))          # 训练协方差(含噪声)/ train covariance
    Ks = rbf_kernel(X,Xs,l,sf)                                # 训练-测试核 / cross covariance
    L  = np.linalg.cholesky(K); alpha=np.linalg.solve(L.T, np.linalg.solve(L,y))
    mu = Ks.T@alpha                                          # 后验均值 / posterior mean
    v  = np.linalg.solve(L, Ks)
    var= np.diag(rbf_kernel(Xs,Xs,l,sf)) - np.sum(v**2,0)    # 后验方差 / posterior variance
    return mu, np.sqrt(np.maximum(var,0))
mu, sd = gp_posterior(l_opt, sf_opt, sn_opt)
print(f"预测不确定性: 数据点处 std≈{sd[np.argmin(abs(Xs+2.5))]:.2f}, 空缺处(x=0.3) std≈{sd[np.argmin(abs(Xs-0.3))]:.2f}, 外推(x=6) std≈{sd[-1]:.2f}")


**中文**：可视化——左图 GP 先验采样(还没看数据),中图 GP 后验(均值+不确定带+后验采样,看不确定性在空缺和外推处张开),右图长度尺度的影响。
**English**: Visualization — left: GP prior samples (before data); middle: GP posterior (mean + uncertainty band + posterior samples, uncertainty widening in the gap and extrapolation); right: the effect of lengthscale.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,4.8))
# ① GP 先验采样 / prior samples
for s in prior_samples: ax[0].plot(Xs, s, alpha=0.7)
ax[0].fill_between(Xs.ravel(), -2, 2, color="gray", alpha=0.1)
ax[0].set_title("GP 先验:从'所有平滑函数'里采样 / prior samples"); ax[0].set_xlabel("x"); ax[0].set_ylabel("f(x)"); ax[0].set_ylim(-3,3)
# ② GP 后验 / posterior
ax[1].scatter(X, y, c="k", zorder=5, label="数据 data")
ax[1].plot(Xs, true_f(Xs), "g--", label="真实 true")
ax[1].plot(Xs, mu, "b", label="后验均值 mean")
ax[1].fill_between(Xs.ravel(), mu-2*sd, mu+2*sd, color="b", alpha=0.2, label="±2σ 不确定")
L=np.linalg.cholesky(rbf_kernel(X,X,l_opt,sf_opt)+sn_opt**2*np.eye(len(X)))
post_cov=rbf_kernel(Xs,Xs,l_opt,sf_opt)-(np.linalg.solve(L,rbf_kernel(X,Xs,l_opt,sf_opt))).T@np.linalg.solve(L,rbf_kernel(X,Xs,l_opt,sf_opt))
for s in np.random.multivariate_normal(mu, post_cov+1e-8*np.eye(len(Xs)), 4): ax[1].plot(Xs, s, "b", alpha=0.25)
ax[1].axvspan(-1,1.7,color="orange",alpha=0.08); ax[1].set_title("GP 后验:空缺&外推处不确定性张开 / uncertainty grows in gaps"); ax[1].set_xlabel("x"); ax[1].legend(fontsize=7); ax[1].set_ylim(-3,3)
# ③ 长度尺度的影响 / lengthscale effect
for l,c in [(0.3,"#C44E52"),(1.0,"#4C72B0"),(3.0,"#55A868")]:
    m,s=gp_posterior(l, sf_opt, sn_opt); ax[2].plot(Xs, m, c, label=f"ℓ={l}")
ax[2].scatter(X,y,c="k",zorder=5); ax[2].set_title("长度尺度 ℓ:小=摆动 大=平滑 / lengthscale"); ax[2].set_xlabel("x"); ax[2].legend(fontsize=8); ax[2].set_ylim(-2,2)
plt.tight_layout(); plt.savefig("/tmp/bay06_viz.png",dpi=80); plt.show()
print("中图关键:数据处不确定带收紧(它见过), 中间空缺&两端外推处张开(它没见过)——GP 诚实地量化无知")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **GP 直接在函数上做贝叶斯,自带诚实的不确定性**:看中图——GP 后验在**有数据处收紧**(它见过、很确定),在**中间空缺(橙色)和两端外推处张开**(它没见过、坦承不确定)。这是 GP 最迷人也最实用的性质:**它知道自己在哪里无知**。而且它**没有假设函数是几次多项式**——形状完全由数据 + 核决定,这就是非参数的自由。
2. **核 = 你对函数的先验假设**:长度尺度 $\ell$ 小,函数摆动快、点之间几乎不共享信息(右图红线);$\ell$ 大,函数平滑、远处点也相互影响(绿线)。选对核(RBF 光滑、Matérn 粗糙、周期核……)就是在**注入领域知识**。而超参不用手调——**最大化边际似然自动选**,还内置"奥卡姆剃刀"(太摆动或太平滑的模型边际似然都低)。
3. **诚实的致命局限:$O(n^3)$**:GP 要对 $n\times n$ 的核矩阵求逆,时间 $O(n^3)$、内存 $O(n^2)$。**几千个点就开始吃力,几万个点就跑不动**。这是 GP 没能像神经网络那样统治大数据的根本原因。解法:**稀疏 GP / 诱导点(inducing points)**、随机特征近似、KISS-GP 等,用少量"代表点"近似完整 GP。所以 GP 的主战场是**小数据 + 高价值 + 需要不确定性**的场景(下节的贝叶斯优化正是如此)。

**English**:
1. **GP does Bayes directly over functions, with honest built-in uncertainty**: see the middle plot — the posterior tightens **where there is data** (seen, confident) and widens **in the gap (orange) and when extrapolating** (unseen, honestly uncertain). This is a GP's most beautiful and useful property: **it knows where it is ignorant.** And it **assumes no polynomial degree** — the shape is set entirely by data + kernel, the freedom of non-parametrics.
2. **The kernel = your prior belief about the function**: small lengthscale $\ell$ → the function wiggles fast, points barely share information (red, right); large $\ell$ → smooth, distant points influence each other (green). Choosing the kernel (RBF smooth, Matérn rough, periodic…) is **injecting domain knowledge**. And you don't hand-tune hyperparameters — **maximizing the marginal likelihood picks them automatically**, with a built-in "Occam's razor" (too wiggly or too smooth both get low marginal likelihood).
3. **The honest fatal limit: $O(n^3)$**: a GP inverts an $n\times n$ kernel matrix — $O(n^3)$ time, $O(n^2)$ memory. **It struggles at a few thousand points and can't run at tens of thousands.** This is why GPs never dominated big data like neural nets. Fixes: **sparse GPs / inducing points**, random-feature approximations, KISS-GP — approximating the full GP with a few "representative points." So GPs' home turf is **small data + high value + needs uncertainty** (exactly Bayesian optimization, next).

> 💼 **实战视角 / Practical angle**
> **中文**:GP 的实战价值:①**小数据回归 + 可靠不确定性**(每个实验都贵的科学/工程);②**贝叶斯优化**(用 GP 的不确定性指导下一次昂贵实验——调超参、材料发现、A/B, 下节 18.7);③**地统计 kriging**(矿产、气象插值);④时序(带周期核)。**工程要点**:①几千点以内直接用(sklearn `GaussianProcessRegressor`、GPyTorch);②大数据用**稀疏 GP**;③核选择是关键先验(不确定就用 Matérn);④输入要标准化。面试金句:*"GP 是函数上的高斯分布, 核定义平滑度; 预测是闭式高斯条件, 数据外方差自动变大=诚实的不确定性; 超参用边际似然选; 但 O(n³) 限制它只适合小数据——所以是贝叶斯优化的完美搭档。"*
> **English**: GP practical value: ① **small-data regression + reliable uncertainty** (science/engineering where each experiment is costly); ② **Bayesian optimization** (use GP uncertainty to guide the next expensive experiment — hyperparameter tuning, materials discovery, A/B — 18.7 next); ③ **geostatistical kriging** (mining, weather interpolation); ④ time series (with a periodic kernel). **Engineering**: ① use directly up to a few thousand points (sklearn `GaussianProcessRegressor`, GPyTorch); ② big data needs **sparse GPs**; ③ kernel choice is a key prior (use Matérn if unsure); ④ standardize inputs. Interview line: *"A GP is a Gaussian distribution over functions, the kernel defines smoothness; prediction is the closed-form Gaussian conditional, with variance auto-growing outside data = honest uncertainty; hyperparameters via marginal likelihood; but O(n³) limits it to small data — making it the perfect partner for Bayesian optimization."*

---
### 小结 / Summary
- **中文**:GP=函数上的分布(任意有限点联合高斯), 由均值+核决定; 核控制平滑度(长度尺度)。
- **English**: GP = a distribution over functions (finite points jointly Gaussian), set by a mean + kernel; the kernel controls smoothness (lengthscale).
- **中文**:预测=闭式高斯条件, 均值+随距离增长的方差(数据外自动不确定); 超参用边际似然选。
- **English**: Prediction = closed-form Gaussian conditional, mean + distance-growing variance (auto-uncertain outside data); hyperparameters via marginal likelihood.
- **中文**:非参数、不确定性精准, 但 $O(n^3)$ 只适合小数据→稀疏GP; 是贝叶斯优化的引擎(下节)。
- **English**: Non-parametric with precise uncertainty, but $O(n^3)$ suits only small data → sparse GPs; the engine of Bayesian optimization (next).
